In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from keras.models import Model
from keras.layers import Input, Embedding, Flatten, Dense, Concatenate
from keras.optimizers import Adam

# Load the data
data_path = "/Users/igormol/Desktop/analytics3/sample/Marajo.csv"
df = pd.read_csv(data_path)

# Fill missing values in the 'Brand_Name' column
df['Brand_Name'].fillna('Unknown', inplace=True)

# Encode categorical variables
client_encoder = LabelEncoder()
product_encoder = LabelEncoder()

df['Client_ID_encoded'] = client_encoder.fit_transform(df['Client_ID'])
df['Product_ID_encoded'] = product_encoder.fit_transform(df['Product_ID'])

# Create interaction matrix (implicit feedback)
interaction_matrix = pd.pivot_table(df, index='Client_ID_encoded', columns='Product_ID_encoded', values='Monetary_Value', fill_value=0)

# Extract interaction pairs
client_ids = df['Client_ID_encoded'].values
product_ids = df['Product_ID_encoded'].values
ratings = df['Monetary_Value'].values

# Split the data into training and testing sets
train_client_ids, test_client_ids, train_product_ids, test_product_ids, train_ratings, test_ratings = train_test_split(
    client_ids, product_ids, ratings, test_size=0.2, random_state=42
)

# Set embedding dimension
embedding_dim = 50

# Client input and embedding
client_input = Input(shape=[1], name='client')
client_embedding = Embedding(len(client_encoder.classes_), embedding_dim, name='client_embedding')(client_input)
client_vec = Flatten(name='client_flatten')(client_embedding)

# Product input and embedding
product_input = Input(shape=[1], name='product')
product_embedding = Embedding(len(product_encoder.classes_), embedding_dim, name='product_embedding')(product_input)
product_vec = Flatten(name='product_flatten')(product_embedding)

# Concatenate embeddings
concat = Concatenate()([client_vec, product_vec])

# Fully connected layers
fc1 = Dense(128, activation='relu')(concat)
fc2 = Dense(64, activation='relu')(fc1)
output = Dense(1)(fc2)

# Build and compile model
model = Model([client_input, product_input], output)
model.compile(optimizer=Adam(lr=0.001), loss='mean_squared_error')

# Train the model
history = model.fit([train_client_ids, train_product_ids], train_ratings, 
                    batch_size=64, epochs=10, validation_split=0.2, verbose=1)

def recall_at_k(actual, predicted, k):
    actual_set = set(actual)
    predicted_set = set(predicted[:k])
    return len(actual_set & predicted_set) / float(len(actual_set))

# Generate recommendations
all_clients = np.array(range(len(client_encoder.classes_)))
all_products = np.array(range(len(product_encoder.classes_)))

recommendations = []

for client_id in all_clients:
    client_vec = np.array([client_id] * len(all_products))
    product_scores = model.predict([client_vec, all_products], verbose=0)
    product_scores = product_scores.flatten()
    
    # Get top 10 recommendations
    top_10_products = np.argsort(product_scores)[-10:][::-1]
    top_10_scores = product_scores[top_10_products]
    
    # Compute recall at k (example with k=10)
    actual_purchased = df[df['Client_ID_encoded'] == client_id]['Product_ID_encoded'].values
    rec_at_k = recall_at_k(actual_purchased, top_10_products, 10)
    
    for product, score in zip(top_10_products, top_10_scores):
        recommendations.append((client_encoder.inverse_transform([client_id])[0], 
                                product_encoder.inverse_transform([product])[0], 
                                score, rec_at_k))

# Convert recommendations to DataFrame
rec_df = pd.DataFrame(recommendations, columns=['Client_ID', 'Product_Name', 'Predicted_Rating', 'Recall_at_k'])
rec_df['Product_Name'] = product_encoder.inverse_transform(rec_df['Product_Name'].astype(int))

# Save the recommendations to a CSV file
output_path = "/Users/igormol/Desktop/analytics3/sample/recommendations.csv"
rec_df.to_csv(output_path, index=False)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from keras.models import Model
from keras.layers import Input, Embedding, Flatten, Dense, Concatenate
from keras.optimizers import Adam

# Load the data
data_path = "/Users/igormol/Desktop/analytics3/sample/Marajo.csv"
df = pd.read_csv(data_path)

# Fill missing values in the 'Brand_Name' column
df['Brand_Name'].fillna('Unknown', inplace=True)

# Encode categorical variables
client_encoder = LabelEncoder()
product_encoder = LabelEncoder()

df['Client_ID_encoded'] = client_encoder.fit_transform(df['Client_ID'])
df['Product_ID_encoded'] = product_encoder.fit_transform(df['Product_ID'])

# Create interaction matrix (implicit feedback)
interaction_matrix = pd.pivot_table(df, index='Client_ID_encoded', columns='Product_ID_encoded', values='Monetary_Value', fill_value=0)

# Extract interaction pairs
client_ids = df['Client_ID_encoded'].values
product_ids = df['Product_ID_encoded'].values
ratings = df['Monetary_Value'].values

# Split the data into training and testing sets
train_client_ids, test_client_ids, train_product_ids, test_product_ids, train_ratings, test_ratings = train_test_split(
    client_ids, product_ids, ratings, test_size=0.2, random_state=42
)

# Set embedding dimension
embedding_dim = 50

# Client input and embedding
client_input = Input(shape=[1], name='client')
client_embedding = Embedding(len(client_encoder.classes_), embedding_dim, name='client_embedding')(client_input)
client_vec = Flatten(name='client_flatten')(client_embedding)

# Product input and embedding
product_input = Input(shape=[1], name='product')
product_embedding = Embedding(len(product_encoder.classes_), embedding_dim, name='product_embedding')(product_input)
product_vec = Flatten(name='product_flatten')(product_embedding)

# Concatenate embeddings
concat = Concatenate()([client_vec, product_vec])

# Fully connected layers
fc1 = Dense(128, activation='relu')(concat)
fc2 = Dense(64, activation='relu')(fc1)
output = Dense(1)(fc2)

# Build and compile model
model = Model([client_input, product_input], output)
model.compile(optimizer=Adam(lr=0.001), loss='mean_squared_error')

# Train the model
history = model.fit([train_client_ids, train_product_ids], train_ratings, 
                    batch_size=64, epochs=10, validation_split=0.2, verbose=1)

# Define recall_at_k function
def recall_at_k(actual, predicted, k):
    actual_set = set(actual)
    predicted_set = set(predicted[:k])
    return len(actual_set & predicted_set) / float(len(actual_set))

# Generate predictions for all client-product pairs in batches
all_clients = np.array(range(len(client_encoder.classes_)))
all_products = np.array(range(len(product_encoder.classes_)))
clients_expanded = np.repeat(all_clients, len(all_products))
products_tiled = np.tile(all_products, len(all_clients))

predictions = model.predict([clients_expanded, products_tiled], verbose=0)
predictions = predictions.flatten().reshape(len(all_clients), len(all_products))

# Generate recommendations
recommendations = []

for client_index, client_id in enumerate(all_clients):
    product_scores = predictions[client_index]
    
    # Get top 10 recommendations
    top_10_products = np.argsort(product_scores)[-10:][::-1]
    top_10_scores = product_scores[top_10_products]
    
    # Compute recall at k (example with k=10)
    actual_purchased = df[df['Client_ID_encoded'] == client_id]['Product_ID_encoded'].values
    rec_at_k = recall_at_k(actual_purchased, top_10_products, 10)
    
    for product, score in zip(top_10_products, top_10_scores):
        recommendations.append((client_encoder.inverse_transform([client_id])[0], 
                                product_encoder.inverse_transform([product])[0], 
                                score, rec_at_k))

# Convert recommendations to DataFrame
rec_df = pd.DataFrame(recommendations, columns=['Client_ID', 'Product_Name', 'Predicted_Rating', 'Recall_at_k'])
rec_df['Product_Name'] = product_encoder.inverse_transform(rec_df['Product_Name'].astype(int))

# Save the recommendations to a CSV file
output_path = "/Users/igormol/Desktop/analytics3/sample/recommendations.csv"
rec_df.to_csv(output_path, index=False)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from keras.models import Model
from keras.layers import Input, Embedding, Flatten, Dense, Concatenate
from keras.optimizers import Adam

# Load the data
data_path = "/Users/igormol/Desktop/analytics3/sample/Marajo.csv"
df = pd.read_csv(data_path)

# Fill missing values in the 'Brand_Name' column
df['Brand_Name'].fillna('Unknown', inplace=True)

# Encode categorical variables
client_encoder = LabelEncoder()
product_encoder = LabelEncoder()

df['Client_ID_encoded'] = client_encoder.fit_transform(df['Client_ID'])
df['Product_ID_encoded'] = product_encoder.fit_transform(df['Product_ID'])

# Create interaction matrix (implicit feedback)
interaction_matrix = pd.pivot_table(df, index='Client_ID_encoded', columns='Product_ID_encoded', values='Monetary_Value', fill_value=0)

# Extract interaction pairs
client_ids = df['Client_ID_encoded'].values
product_ids = df['Product_ID_encoded'].values
ratings = df['Monetary_Value'].values

# Split the data into training and testing sets
train_client_ids, test_client_ids, train_product_ids, test_product_ids, train_ratings, test_ratings = train_test_split(
    client_ids, product_ids, ratings, test_size=0.2, random_state=42
)

# Set embedding dimension
embedding_dim = 50

# Client input and embedding
client_input = Input(shape=[1], name='client')
client_embedding = Embedding(len(client_encoder.classes_), embedding_dim, name='client_embedding')(client_input)
client_vec = Flatten(name='client_flatten')(client_embedding)

# Product input and embedding
product_input = Input(shape=[1], name='product')
product_embedding = Embedding(len(product_encoder.classes_), embedding_dim, name='product_embedding')(product_input)
product_vec = Flatten(name='product_flatten')(product_embedding)

# Concatenate embeddings
concat = Concatenate()([client_vec, product_vec])

# Fully connected layers
fc1 = Dense(128, activation='relu')(concat)
fc2 = Dense(64, activation='relu')(fc1)
output = Dense(1)(fc2)

# Build and compile model
model = Model([client_input, product_input], output)
model.compile(optimizer=Adam(lr=0.001), loss='mean_squared_error')

# Train the model
history = model.fit([train_client_ids, train_product_ids], train_ratings, 
                    batch_size=64, epochs=1, validation_split=0.2, verbose=1)

# Define recall_at_k function
def recall_at_k(actual, predicted, k):
    actual_set = set(actual)
    predicted_set = set(predicted[:k])
    return len(actual_set & predicted_set) / float(len(actual_set))

# Generate predictions for all client-product pairs in batches
all_clients = np.array(range(len(client_encoder.classes_)))
all_products = np.array(range(len(product_encoder.classes_)))
clients_expanded = np.repeat(all_clients, len(all_products))
products_tiled = np.tile(all_products, len(all_clients))

predictions = model.predict([clients_expanded, products_tiled], verbose=0)
predictions = predictions.flatten().reshape(len(all_clients), len(all_products))

# Generate recommendations
recommendations = []

for client_index, client_id in enumerate(all_clients):
    product_scores = predictions[client_index]
    
    # Get top 10 recommendations
    top_10_products = np.argsort(product_scores)[-10:][::-1]
    top_10_scores = product_scores[top_10_products]
    
    # Compute recall at k (example with k=10)
    actual_purchased = df[df['Client_ID_encoded'] == client_id]['Product_ID_encoded'].values
    rec_at_k = recall_at_k(actual_purchased, top_10_products, 10)
    
    for product, score in zip(top_10_products, top_10_scores):
        recommendations.append((client_id, product, score, rec_at_k))

# Convert recommendations to DataFrame
rec_df = pd.DataFrame(recommendations, columns=['Client_ID_encoded', 'Product_ID_encoded', 'Predicted_Rating', 'Recall_at_k'])

# Decode client and product IDs
rec_df['Client_ID'] = client_encoder.inverse_transform(rec_df['Client_ID_encoded'])
rec_df['Product_Name'] = product_encoder.inverse_transform(rec_df['Product_ID_encoded'])

# Select required columns
rec_df = rec_df[['Client_ID', 'Product_Name', 'Predicted_Rating', 'Recall_at_k']]

# Save the recommendations to a CSV file
output_path = "/Users/igormol/Desktop/analytics3/sample/recommendations.csv"
rec_df.to_csv(output_path, index=False)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import recall_score

# Load the data
data_path = "/Users/igormol/Desktop/analytics3/sample/Marajo.csv"
data = pd.read_csv(data_path)

# Fill missing values (e.g., brand names) if necessary
data['Brand_Name'].fillna('Unknown', inplace=True)

# Encode Client_ID and Product_ID
client_encoder = LabelEncoder()
product_encoder = LabelEncoder()

data['Client_ID'] = client_encoder.fit_transform(data['Client_ID'])
data['Product_ID'] = product_encoder.fit_transform(data['Product_ID'])

# Extract features and target
X = data[['Client_ID', 'Product_ID']]
y = data['Monetary_Value']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the NCF model
num_clients = data['Client_ID'].nunique()
num_products = data['Product_ID'].nunique()

# Embedding dimension
embedding_dim = 50

# Client embedding
client_input = keras.layers.Input(shape=(1,), name='client_input')
client_embedding = keras.layers.Embedding(input_dim=num_clients, output_dim=embedding_dim, name='client_embedding')(client_input)
client_vec = keras.layers.Flatten()(client_embedding)

# Product embedding
product_input = keras.layers.Input(shape=(1,), name='product_input')
product_embedding = keras.layers.Embedding(input_dim=num_products, output_dim=embedding_dim, name='product_embedding')(product_input)
product_vec = keras.layers.Flatten()(product_embedding)

# Concatenate embeddings
concat = keras.layers.Concatenate()([client_vec, product_vec])

# Fully connected layers
dense1 = keras.layers.Dense(128, activation='relu')(concat)
dense2 = keras.layers.Dense(64, activation='relu')(dense1)
dense3 = keras.layers.Dense(32, activation='relu')(dense2)
output = keras.layers.Dense(1)(dense3)

# Define the model
model = keras.models.Model(inputs=[client_input, product_input], outputs=output)
model.compile(optimizer='adam', loss='mse')

# Train the model
history = model.fit([X_train['Client_ID'], X_train['Product_ID']], y_train, epochs=10, batch_size=64, validation_split=0.2)

# Predictions
y_pred = model.predict([X_test['Client_ID'], X_test['Product_ID']])

# Compute recall at k
def recall_at_k(y_true, y_pred, k=10):
    relevant_items = np.argsort(-y_true)[:k]
    recommended_items = np.argsort(-y_pred)[:k]
    recall = len(set(relevant_items) & set(recommended_items)) / len(relevant_items)
    return recall

# Compute recall for the test set
recalls = []
for client_id in X_test['Client_ID'].unique():
    client_data = X_test[X_test['Client_ID'] == client_id]
    y_true = y_test[client_data.index]
    y_pred_client = model.predict([client_data['Client_ID'], client_data['Product_ID']])
    recalls.append(recall_at_k(y_true, y_pred_client))

average_recall = np.mean(recalls)
print(f'Average Recall at 10: {average_recall}')

# Generate top 10 recommendations for all clients
client_ids = data['Client_ID'].unique()
product_ids = data['Product_ID'].unique()

recommendations = []

for client_id in client_ids:
    client_data = pd.DataFrame({'Client_ID': client_id, 'Product_ID': product_ids})
    predicted_ratings = model.predict([client_data['Client_ID'], client_data['Product_ID']])
    client_data['Predicted_Rating'] = predicted_ratings
    top_10 = client_data.sort_values(by='Predicted_Rating', ascending=False).head(10)
    
    for _, row in top_10.iterrows():
        product_name = data[data['Product_ID'] == row['Product_ID']]['Product_Name'].values[0]
        recommendations.append({
            'Client_ID': client_encoder.inverse_transform([row['Client_ID']])[0],
            'Product_Name': product_name,
            'Predicted_Rating': row['Predicted_Rating'],
            'Recall_at_k': recall_at_k(y_test, row['Predicted_Rating'])
        })

recommendations_df = pd.DataFrame(recommendations)

# Save the recommendations to a CSV file
output_path = "/Users/igormol/Desktop/analytics3/sample/recommendations.csv"
recommendations_df.to_csv(output_path, index=False)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Load the data
data_path = "/Users/igormol/Desktop/analytics3/sample/Marajo.csv"
data = pd.read_csv(data_path)

# Fill missing values (e.g., brand names) if necessary
data['Brand_Name'].fillna('Unknown', inplace=True)

# Encode Client_ID and Product_ID
client_encoder = LabelEncoder()
product_encoder = LabelEncoder()

data['Client_ID'] = client_encoder.fit_transform(data['Client_ID'])
data['Product_ID'] = product_encoder.fit_transform(data['Product_ID'])

# Extract features and target
X = data[['Client_ID', 'Product_ID']]
y = data['Monetary_Value']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the NCF model
num_clients = data['Client_ID'].nunique()
num_products = data['Product_ID'].nunique()

# Embedding dimension
embedding_dim = 50

# Client embedding
client_input = keras.layers.Input(shape=(1,), name='client_input')
client_embedding = keras.layers.Embedding(input_dim=num_clients, output_dim=embedding_dim, name='client_embedding')(client_input)
client_vec = keras.layers.Flatten()(client_embedding)

# Product embedding
product_input = keras.layers.Input(shape=(1,), name='product_input')
product_embedding = keras.layers.Embedding(input_dim=num_products, output_dim=embedding_dim, name='product_embedding')(product_input)
product_vec = keras.layers.Flatten()(product_embedding)

# Concatenate embeddings
concat = keras.layers.Concatenate()([client_vec, product_vec])

# Fully connected layers
dense1 = keras.layers.Dense(128, activation='relu')(concat)
dense2 = keras.layers.Dense(64, activation='relu')(dense1)
dense3 = keras.layers.Dense(32, activation='relu')(dense2)
output = keras.layers.Dense(1)(dense3)

# Define the model
model = keras.models.Model(inputs=[client_input, product_input], outputs=output)
model.compile(optimizer='adam', loss='mse')

# Train the model
history = model.fit([X_train['Client_ID'], X_train['Product_ID']], y_train, epochs=10, batch_size=64, validation_split=0.2)

# Predictions
y_pred = model.predict([X_test['Client_ID'], X_test['Product_ID']])

# Compute recall at k
def recall_at_k(y_true, y_pred, k=10):
    relevant_items = list(np.argsort(-y_true)[:k])
    recommended_items = list(np.argsort(-y_pred)[:k])
    recall = len(set(relevant_items) & set(recommended_items)) / len(relevant_items)
    return recall

# Compute recall for the test set
recalls = []
for client_id in X_test['Client_ID'].unique():
    client_data = X_test[X_test['Client_ID'] == client_id]
    y_true = y_test[client_data.index].values
    y_pred_client = model.predict([client_data['Client_ID'], client_data['Product_ID']]).flatten()
    recalls.append(recall_at_k(y_true, y_pred_client))

average_recall = np.mean(recalls)
print(f'Average Recall at 10: {average_recall}')

# Generate top 10 recommendations for all clients
client_ids = data['Client_ID'].unique()
product_ids = data['Product_ID'].unique()

recommendations = []

for client_id in client_ids:
    client_data = pd.DataFrame({'Client_ID': client_id, 'Product_ID': product_ids})
    predicted_ratings = model.predict([client_data['Client_ID'], client_data['Product_ID']]).flatten()
    client_data['Predicted_Rating'] = predicted_ratings
    top_10 = client_data.sort_values(by='Predicted_Rating', ascending=False).head(10)
    
    for _, row in top_10.iterrows():
        product_name = data[data['Product_ID'] == row['Product_ID']]['Product_Name'].values[0]
        recommendations.append({
            'Client_ID': client_encoder.inverse_transform([row['Client_ID']])[0],
            'Product_Name': product_name,
            'Predicted_Rating': row['Predicted_Rating'],
            'Recall_at_k': recall_at_k(y_test, row['Predicted_Rating'])
        })

recommendations_df = pd.DataFrame(recommendations)

# Save the recommendations to a CSV file
output_path = "/Users/igormol/Desktop/analytics3/sample/recommendations.csv"
recommendations_df.to_csv(output_path, index=False)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Load the data
data_path = "/Users/igormol/Desktop/analytics3/sample/Marajo.csv"
data = pd.read_csv(data_path)

# Fill missing values (e.g., brand names) if necessary
data['Brand_Name'].fillna('Unknown', inplace=True)

# Encode Client_ID and Product_ID
client_encoder = LabelEncoder()
product_encoder = LabelEncoder()

data['Client_ID'] = client_encoder.fit_transform(data['Client_ID'])
data['Product_ID'] = product_encoder.fit_transform(data['Product_ID'])

# Extract features and target
X = data[['Client_ID', 'Product_ID']]
y = data['Monetary_Value']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the NCF model
num_clients = data['Client_ID'].nunique()
num_products = data['Product_ID'].nunique()

# Embedding dimension
embedding_dim = 50

# Client embedding
client_input = keras.layers.Input(shape=(1,), name='client_input')
client_embedding = keras.layers.Embedding(input_dim=num_clients, output_dim=embedding_dim, name='client_embedding')(client_input)
client_vec = keras.layers.Flatten()(client_embedding)

# Product embedding
product_input = keras.layers.Input(shape=(1,), name='product_input')
product_embedding = keras.layers.Embedding(input_dim=num_products, output_dim=embedding_dim, name='product_embedding')(product_input)
product_vec = keras.layers.Flatten()(product_embedding)

# Concatenate embeddings
concat = keras.layers.Concatenate()([client_vec, product_vec])

# Fully connected layers
dense1 = keras.layers.Dense(128, activation='relu')(concat)
dense2 = keras.layers.Dense(64, activation='relu')(dense1)
dense3 = keras.layers.Dense(32, activation='relu')(dense2)
output = keras.layers.Dense(1)(dense3)

# Define the model
model = keras.models.Model(inputs=[client_input, product_input], outputs=output)
model.compile(optimizer='adam', loss='mse')

# Train the model
history = model.fit([X_train['Client_ID'], X_train['Product_ID']], y_train, epochs=1, batch_size=64, validation_split=0.2)

# Predictions
y_pred = model.predict([X_test['Client_ID'], X_test['Product_ID']])

# Compute recall at k
def recall_at_k(y_true, y_pred, k=10):
    relevant_items = list(np.argsort(-y_true)[:k])
    recommended_items = list(np.argsort(-y_pred)[:k])
    recall = len(set(relevant_items) & set(recommended_items)) / len(relevant_items)
    return recall

# Compute recall for the test set
recalls = []
for client_id in X_test['Client_ID'].unique():
    client_data = X_test[X_test['Client_ID'] == client_id]
    y_true = y_test[client_data.index].values
    y_pred_client = model.predict([client_data['Client_ID'], client_data['Product_ID']]).flatten()
    recalls.append(recall_at_k(y_true, y_pred_client))

average_recall = np.mean(recalls)
print(f'Average Recall at 10: {average_recall}')

# Generate mappings from encoded client IDs to original client IDs
client_id_mapping = {encoded_id: original_id for encoded_id, original_id in zip(data['Client_ID'], client_encoder.inverse_transform(data['Client_ID']))}

# Generate top 10 recommendations for all clients
client_ids = data['Client_ID'].unique()
product_ids = data['Product_ID'].unique()

recommendations = []

for client_id in client_ids:
    client_data = pd.DataFrame({'Client_ID': client_id, 'Product_ID': product_ids})
    predicted_ratings = model.predict([client_data['Client_ID'], client_data['Product_ID']]).flatten()
    client_data['Predicted_Rating'] = predicted_ratings
    top_10 = client_data.sort_values(by='Predicted_Rating', ascending=False).head(10)
    
    for _, row in top_10.iterrows():
        product_name = data[data['Product_ID'] == row['Product_ID']]['Product_Name'].values[0]
        recommendations.append({
            'Client_ID': client_id_mapping[int(row['Client_ID'])],  # Ensure Client_ID is an integer
            'Product_Name': product_name,
            'Predicted_Rating': row['Predicted_Rating'],
            'Recall_at_k': recall_at_k(y_test, row['Predicted_Rating'])
        })

recommendations_df = pd.DataFrame(recommendations)

# Save the recommendations to a CSV file
output_path = "/Users/igormol/Desktop/analytics3/sample/recommendations.csv"
recommendations_df.to_csv(output_path, index=False)
